In [95]:
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [221]:


candidates = []

with open("../data/candidates.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        candidates.append(json.loads(line))

In [222]:
skill_weights = {
    "NLP":5,
    "Fine-tuning LLMs":5,
    "LoRA":5,
    "Speech Recognition":5,
    "TTS":5,
    "Image Classification":4,
    "Object Detection":4,
    "GANs":4,
    "Milvus":3,
    "BentoML":3,
    "Weights & Biases":3,
    "Apache Beam":3,
    "Apache Flink":3,
    "Airflow":3,
    "Databricks":3,
    "AWS":2,
    "GCP":2,
    "SQL":2,
    "Flask":2,
    "Kubernetes":2,
    "Terraform":2,
    "Django":2,
    "gRPC":2,
}

In [223]:
def behavior_score(candidate):
    score = 0

    # recruiter_response_rate
    rr = candidate["redrob_signals"]["recruiter_response_rate"]

    if rr > 0.8:
     score += 2
    elif rr > 0.5:
     score += 1
    elif rr > 0.2:
     score += 0.5
        
    # open to work
    if candidate["redrob_signals"]["open_to_work_flag"]:
        score += 2
        
    #notice_period_days
    notice = candidate["redrob_signals"]["notice_period_days"]

    if notice <= 30:
     score += 3
    elif notice <= 60:
     score += 2
    elif notice <= 90:
     score += 1

    #github_activity_score
    github = candidate["redrob_signals"]["github_activity_score"]
    if github == -1:
      score += 0
    elif github < 10:
      score += 1
    elif github < 30:
      score += 2
    else:
      score += 3
    return score

In [224]:
print(behavior_score(candidates[0]))

5.5


In [225]:
title_weights = {
    "ai engineer": 5,
    "ml engineer": 5,
    "machine learning engineer": 5,
    "recommendation systems engineer": 5,
    "recommendation engineer": 5,

    "data scientist": 4,
    "research scientist": 4,
    "applied scientist": 4,

    "data engineer": 3,
    "mlops engineer": 3,

    "backend engineer": 2,
    "software engineer": 2,
    "full stack developer": 2,

    "cloud engineer": 1,
    "devops engineer": 1
}

In [226]:
def title_score(candidate):

    title = candidate["career_history"][0]["title"].lower()

    score = 0

    for role, weight in title_weights.items():
        if role in title:
            score = max(score, weight)

    return score

In [227]:
print(title_score(candidates[0]))

2


In [228]:
print(candidates[0]["career_history"][0]["title"])

Backend Engineer


In [229]:
description_weights = {

    # Core AI/ML
    "machine learning": 5,
    "deep learning": 5,
    "llm": 5,
    "large language model": 5,
    "fine-tuning": 5,
    "lora": 5,
    "rag": 5,
    "embeddings": 5,

    # NLP/CV
    "nlp": 4,
    "transformer": 4,
    "bert": 4,
    "gpt": 4,
    "computer vision": 4,
    "image classification": 4,
    "object detection": 4,

    # ML Engineering
    "mlops": 4,
    "feature engineering": 4,
    "feature pipeline": 4,
    "model deployment": 4,
    "model serving": 4,

    # Data Engineering
    "spark": 3,
    "kafka": 3,
    "airflow": 3,
    "beam": 3,
    "flink": 3,
    "databricks": 3,
    "etl": 3,

    # Libraries
    "tensorflow": 3,
    "pytorch": 3,
    "scikit-learn": 3,
    "xgboost": 3,
    "hugging face": 3,

    # Supporting
    "aws": 2,
    "gcp": 2,
    "azure": 2,
    "docker": 2,
    "kubernetes": 2,
}

In [230]:
def description_score(candidate):

    description = candidate["career_history"][0]["description"].lower()

    score = 0

    matched = set()

    for word, weight in description_weights.items():

      if word in description and word not in matched:
        score += weight
        matched.add(word)

    return min(score, 12)

In [231]:
description_score(candidates[31])

8

In [232]:
print(description_score(candidates[0]))
print(candidates[1]["career_history"][0]["description"][:200])

10
Customer support team lead at a SaaS product. Managed a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to produ


In [233]:
def skills_score(candidate):

    skill_scores = []

    for skill in candidate["skills"]:

        skill_score = 0

        name = skill["name"]

        if name in skill_weights:

            # Base score for relevance
            skill_score += skill_weights[name]

            # Proficiency
            proficiency = skill["proficiency"]

            if proficiency == "advanced":
                skill_score += 1
            elif proficiency == "intermediate":
                skill_score += 0.5
            elif proficiency == "beginner":
                skill_score += 0

            # Duration
            duration = skill["duration_months"]

            if duration >= 36:
                skill_score += 0.5
            elif duration >= 12:
                skill_score += 0.25

            # Endorsements
            endorsements = skill["endorsements"]

            if endorsements >= 20:
                skill_score += 0.5
            elif endorsements >= 5:
                skill_score += 0.25

            # Store this skill's score
            skill_scores.append(skill_score)

    # Highest scoring skills first
    skill_scores.sort(reverse=True)

    return min(sum(skill_scores[:3]), 15)


In [108]:
print(skills_score(candidates[0]))
print(skills_score(candidates[9]))

15
15


In [237]:
def candidate_to_text(candidate):

    parts = []

    # Current title (repeat once for extra importance)
    parts.append(candidate["career_history"][0]["title"])
    parts.append(candidate["career_history"][0]["title"])

    # Company
    parts.append(candidate["career_history"][0]["company"])

    # Description
    parts.append(candidate["career_history"][0]["description"])

    # Skills
    for skill in candidate["skills"]:
        parts.append(skill["name"])

    return " ".join(parts).lower()

In [238]:
print(candidate_to_text(candidates[0])[:500])

backend engineer backend engineer mindtree implemented streaming data pipelines on kafka and spark streaming for a real-time user-activity processing platform. designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. worked closely with the data science team to make sure feature pipelines aligned with what their models needed. most of my career has been data engineering, with some adjacent ml exposure. tailwind nlp i


In [239]:
print(type(candidate_to_text(candidates[0])))

<class 'str'>


# tf-idf

In [240]:
job_description = """
Looking for an AI/ML Engineer with experience in
Machine Learning,
Deep Learning,
Natural Language Processing,
Large Language Models,
Fine-tuning LLMs,
RAG,
Embeddings,
Vector Databases,
Information Retrieval,
Python,
PyTorch,
TensorFlow,
Hugging Face,
MLOps,
Feature Engineering,
Model Deployment,
Recommendation Systems.
"""

In [241]:
documents = [job_description]

for candidate in candidates:
    documents.append(candidate_to_text(candidate))

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    min_df=2
)

tfidf_matrix = vectorizer.fit_transform(documents)

In [243]:
print(tfidf_matrix.shape)

(100001, 1008)


In [244]:
jd_vector = tfidf_matrix[0]
candidate_vectors = tfidf_matrix[1:]

In [245]:
from sklearn.metrics.pairwise import cosine_similarity

candidate_vectors = tfidf_matrix[1:]

scores = cosine_similarity(
    candidate_vectors,
    jd_vector
).flatten()

In [246]:
print(len(scores))

100000


In [247]:
tfidf_scores = {}

for candidate, score in zip(candidates, scores):
    tfidf_scores[id(candidate)] = score

In [248]:
def tfidf_score(candidate):
    return tfidf_scores[id(candidate)]

In [279]:
def penalty_score(candidate):

    title = candidate["career_history"][0]["title"].lower()

    bad_titles = {
        "accountant": 3,
        "mechanical engineer": 3,
        "civil engineer": 3,
        "graphic designer": 3,
        "customer support": 3,
        "marketing manager": 3,
        "operations manager": 2,
        "hr manager": 2,
    }

    penalty = 0

    for bad, value in bad_titles.items():
        if bad in title:
            penalty += value

    return penalty

In [280]:
def rule_score(candidate):

    score = 0

    score += 0.10 * behavior_score(candidate)
    score += 0.15 * title_score(candidate)
    score += 0.35 * description_score(candidate)
    score += 0.40 * skills_score(candidate)

    return score

In [281]:
def score_candidate(candidate):

    final = rule_score(candidate)

    tfidf = tfidf_score(candidate)

    if tfidf >= 0.18:
        final += 1
    elif tfidf >= 0.12:
        final += 0.5

    final -= penalty_score(candidate)

    return final

In [282]:
print(score_candidate(candidates[0]))
print(score_candidate(candidates[1]))
print(score_candidate(candidates[2]))

10.35
-0.34999999999999987
-0.4500000000000002


In [283]:
ranked = []

scoring_cache = {}

for candidate in candidates:

    cid = candidate["candidate_id"]

    if cid not in scoring_cache:
        scoring_cache[cid] = score_candidate(candidate)

    ranked.append(
        (scoring_cache[cid], candidate)
    )

ranked.sort(reverse=True, key=lambda x: x[0])

In [284]:
top100 = ranked[:100]

In [285]:
import pandas as pd

submission = []

for rank, (score, candidate) in enumerate(top100, start=1):

    submission.append({
        "rank": rank,
        "candidate_id": candidate["candidate_id"],
        "score": round(score, 4)
    })

submission_df = pd.DataFrame(submission)

submission_df.head()

,rank,candidate_id,score
0,1,CAND_0066690,12.95
1,2,CAND_0032527,12.75
2,3,CAND_0087494,12.70
3,4,CAND_0018094,12.65
4,5,CAND_0031752,12.65


In [286]:
def generate_reasoning(candidate):

    title = candidate["career_history"][0]["title"]

    skills = [s["name"] for s in candidate["skills"] if s["name"] in skill_weights]
    top_skills = skills[:3]

    tfidf = tfidf_score(candidate)

    tfidf_level = (
        "very high" if tfidf > 0.18 else
        "moderate" if tfidf > 0.12 else
        "low"
    )

    penalty = penalty_score(candidate)

    reason = (
        f"{title} with strong expertise in {', '.join(top_skills)}. "
        f"Semantic relevance is {tfidf_level} based on job description match. "
        f"Penalty adjustment applied: {penalty:.1f} due to role bias filtering. "
        f"Final ranking reflects combined structured skills + semantic alignment."
    )

    return reason

In [287]:
import pandas as pd
submission = []

for rank, (score, candidate) in enumerate(top100, start=1):

    submission.append({
        "rank": rank,
        "candidate_id": candidate["candidate_id"],
        "score": round(score, 4),
        "reasoning": generate_reasoning(candidate)
    })

submission_df = pd.DataFrame(submission)

In [288]:
submission_df.head(3)

,rank,candidate_id,score,reasoning
0,1,CAND_0066690,12.95,ML Engineer with strong expertise in Object De...
1,2,CAND_0032527,12.75,Junior ML Engineer with strong expertise in Ap...
2,3,CAND_0087494,12.70,"ML Engineer with strong expertise in NLP, Fine..."


In [289]:
submission_df.to_csv("submission.csv", index=False)

In [290]:
submission_df.head()

,rank,candidate_id,score,reasoning
0,1,CAND_0066690,12.95,ML Engineer with strong expertise in Object De...
1,2,CAND_0032527,12.75,Junior ML Engineer with strong expertise in Ap...
2,3,CAND_0087494,12.70,"ML Engineer with strong expertise in NLP, Fine..."
3,4,CAND_0018094,12.65,ML Engineer with strong expertise in Speech Re...
4,5,CAND_0031752,12.65,ML Engineer with strong expertise in Object De...


In [291]:
for i in range(3):
    row = submission_df.iloc[i]
    print("="*50)
    print(f"Rank: {row['rank']}")
    print(f"ID: {row['candidate_id']}")
    print(f"Score: {row['score']}")
    print(f"Reasoning: {row['reasoning']}")

Rank: 1
ID: CAND_0066690
Score: 12.95
Reasoning: ML Engineer with strong expertise in Object Detection, Speech Recognition, TTS. Semantic relevance is very high based on job description match. Penalty adjustment applied: 0.0 due to role bias filtering. Final ranking reflects combined structured skills + semantic alignment.
Rank: 2
ID: CAND_0032527
Score: 12.75
Reasoning: Junior ML Engineer with strong expertise in Apache Flink, NLP, TTS. Semantic relevance is very high based on job description match. Penalty adjustment applied: 0.0 due to role bias filtering. Final ranking reflects combined structured skills + semantic alignment.
Rank: 3
ID: CAND_0087494
Score: 12.7
Reasoning: ML Engineer with strong expertise in NLP, Fine-tuning LLMs, TTS. Semantic relevance is very high based on job description match. Penalty adjustment applied: 0.0 due to role bias filtering. Final ranking reflects combined structured skills + semantic alignment.


In [292]:
import numpy as np

scores_only = [s for s, _ in ranked]

print("Min:", np.min(scores_only))
print("Max:", np.max(scores_only))
print("Mean:", np.mean(scores_only))
print("Median:", np.median(scores_only))

Min: -3.0
Max: 12.95
Mean: 2.16599
Median: 1.9


In [293]:
for i in range(10):
    score, cand = ranked[i]
    print(i+1, cand["career_history"][0]["title"], score)

1 ML Engineer 12.95
2 Junior ML Engineer 12.75
3 ML Engineer 12.7
4 ML Engineer 12.649999999999999
5 ML Engineer 12.649999999999999
6 ML Engineer 12.649999999999999
7 ML Engineer 12.6
8 Data Scientist 12.6
9 ML Engineer 12.55
10 ML Engineer 12.549999999999999
